# 06 · Web Scraping de SensaCine

**Etapa 2 del pipeline del Proyecto 2:** enriquecer el corpus con reseñas **escritas originalmente en español**.

- **Fuente:** críticas de espectadores de [SensaCine](https://www.sensacine.com).
- **Salida:** `data/processed/resenas_scraping.csv`.
- **Código:** `src/scraper.py`, siguiendo el patrón del notebook guía `WebScraping.ipynb`.

### ¿Por qué SensaCine?

Se revisaron los cuatro sitios sugeridos en el enunciado:

| Sitio | Resultado |
|---|---|
| FilmAffinity | Responde **403 (prohibido)** incluso al pedir `robots.txt` |
| Letterboxd | Responde **403** en las páginas de reseñas; además, la mayoría están en inglés |
| IMDb | `robots.txt` prohíbe todo (`Disallow: /`) y las reseñas están en inglés |
| **SensaCine** | `robots.txt` **permite** las críticas; el HTML es estático e incluye la calificación en estrellas |

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import scraper as sc

print("Salida:", sc.RUTA_SALIDA.relative_to(RAIZ))

Salida: data\processed\resenas_scraping.csv


## 1. Buenas prácticas y ética

Se cumplen las buenas prácticas del notebook guía y del enunciado (sección 2.2):

| Práctica | Cómo se aplica |
|---|---|
| Revisar `robots.txt` | `sc.check_robots()` (abajo) |
| Términos de uso | Revisados manualmente antes de ejecutar (ver nota) |
| `time.sleep()` entre requests | **2 segundos** después de cada request (el enunciado pide mínimo 1–2 s) |
| User-Agent identificatorio | `CUC-DataMining-Bot/1.0 (Educational Purpose; curso Mineria de Textos)` |
| `timeout` | 15 s por request |
| Manejo de errores HTTP | Reintentos con backoff exponencial ante errores de red, 429 y 5xx (decorador `retry_on_failure` del notebook guía) |
| Bloqueo del sitio | Ante un 403 **se detiene** el scraping; no se intenta evadir |
| Guardar el progreso | El CSV y `scraping_estado.json` se actualizan después de cada película; al volver a ejecutar se continúa sin repetir requests |
| Datos personales | **No se guardan nombres de usuario**: `user_id` queda vacío |
| Uso de los datos | Solo con fines educativos; se registra la fuente (`url_fuente`) y la fecha de recolección (`fecha_recopilacion`) |


In [2]:
permisos = sc.check_robots()
for ruta, permitido in permisos.items():
    print(f"  {'[OK]' if permitido else '[BLOQUEADO]'}: {ruta}")

assert all(permisos.values()), "robots.txt no permite alguna de las rutas: no se debe scrapear"

  [OK]: /peliculas/todas-peliculas/genero-13009/decada-2000/
  [OK]: /peliculas/pelicula-10568/criticas-espectadores/


## 2. Estrategia de extracción

1. Para cada uno de los 5 géneros del Proyecto 1 se recorren los listados de películas de SensaCine por década (`/peliculas/todas-peliculas/genero-<id>/decada-<año>/`), con paginación.
2. De cada película que tenga críticas de usuarios se leen las primeras **3 páginas** de críticas (10 por página). Así ninguna película domina el corpus.
3. Se asigna el género con **el mismo criterio del Proyecto 1**: el primer género de la película que esté entre Drama, Acción, Animación, Terror y Comedia.
4. Se detiene cada género al llegar a su meta.

SensaCine oculta muchos enlaces (por ejemplo, el botón *Siguiente*) en una clase CSS que es la URL en base64; `sc.url_ofuscada()` la decodifica.

### Filtros aplicados a cada reseña

| Filtro | Motivo |
|---|---|
| Calificación ≥ 0,5 estrellas | Toda reseña debe tener calificación numérica (las de 0 estrellas quedarían fuera de la escala 1-10) |
| Entre 30 y 400 palabras | Mismo rango que el corpus del Proyecto 1 |
| Año de la reseña ≥ 1998 | Mismo rango temporal del Proyecto 1 |
| Texto no repetido | Evitar duplicados |

### Formato del CSV

Las primeras 14 columnas son **las mismas y en el mismo orden que `resenas_es.csv`**; al final van las columnas del esquema de MongoDB que `resenas_es.csv` no tiene.

| Columna | Valor en las reseñas de SensaCine |
|---|---|
| `id` | Continúa después del último `id` de `resenas_es.csv` |
| `movie_id` | `sensacine-<id de la película>` |
| `user_id`, `resumen_en`, `texto_en` | Vacíos (no se guardan autores; no hay texto en inglés) |
| `fecha` | Fecha de la reseña, con el mismo formato que `resenas_es.csv` (`24 December 2012`) |
| `calificacion` | Estrellas × 2 (escala 1-10, sección 2.3 del enunciado) |
| `spoiler` | `True` si la reseña tiene texto marcado como spoiler |
| `texto_es`, `n_palabras` | Texto original en español y su número de palabras |
| `genero` | Género asignado, con los mismos nombres de `resenas_es.csv` (Action, Horror, …) |
| `release_date` | Fecha de estreno de la película |
| `anio`, `periodo` | Año de la reseña; `1998-2009` o `2010-2018` (desde 2010 todo va al segundo periodo) |
| `titulo_obra`, `tipo_obra` | Título de la película y `pelicula` |
| `generos` | Todos los géneros de la película en SensaCine, separados por `\|` |
| `fuente`, `url_fuente`, `fecha_recopilacion` | `sensacine`, la página de críticas y la fecha y hora del scraping |

## 3. Configuración

- `METAS_POR_GENERO`: reseñas por género. Con 400 × 5 géneros se obtienen exactamente **2 000** reseñas (el enunciado pide mínimo 100 por género). Cada género se detiene justo al llegar a su meta, incluso a mitad de una película.
- `LIMITE_PRUEBA`: número máximo de requests. Con un número pequeño (por ejemplo 10) se hace una prueba rápida; con `None` se ejecuta el scraping completo.

Cada película aporta unas 15 reseñas válidas con 2–3 requests, así que el scraping completo hace unos **400 requests (~15-20 minutos)** con la pausa de 2 s.

In [37]:
METAS_POR_GENERO = 400   # 400 x 5 géneros = 2 000 reseñas
LIMITE_PRUEBA = 10       # None = scraping completo

scraper = sc.SensaCineScraper(
    delay=2.0,                 # segundos entre requests
    metas_por_genero=METAS_POR_GENERO,
    max_paginas_criticas=3,    # páginas de críticas por película
    min_palabras=30,
    max_palabras=400,
)
print(f"Reseñas ya scrapeadas: {scraper.total()} · siguiente id: {scraper.siguiente_id}")

Reseñas ya scrapeadas: 2000 · siguiente id: 17001


## 4. Ejecución (reanudable)

Si se detiene (o se interrumpe el kernel), basta con volver a ejecutar las celdas de las secciones 3 y 4: continúa donde quedó.

In [38]:
df = scraper.run(max_requests=LIMITE_PRUEBA)
print(f"{len(df)} reseñas en {sc.RUTA_SALIDA.relative_to(RAIZ)}")
df.head()

Iniciando SensaCineScraper
Meta por género: 400 | ya scrapeadas: 2000

  Action: meta cumplida (400)
  Animation: meta cumplida (400)
  Comedy: meta cumplida (400)
  Drama: meta cumplida (400)
  Horror: meta cumplida (400)

Requests hechos: 0 | reseñas totales: 2000
2000 reseñas en data\processed\resenas_scraping.csv


,id,movie_id,user_id,fecha,calificacion,spoiler,resumen_en,texto_en,texto_es,n_palabras,genero,release_date,anio,periodo,titulo_obra,tipo_obra,generos,fuente,url_fuente,fecha_recopilacion
0,15001,sensacine-232669,NaN,26 April 2019,10,False,NaN,NaN,El broche final para este viaje de once años. ...,42,Action,2019-04-25,2019,2010-2018,Vengadores: Endgame,pelicula,Acción|Aventura|Ciencia ficción,sensacine,https://www.sensacine.com/peliculas/pelicula-2...,2026-10-31T14:03:55
1,15002,sensacine-232669,NaN,28 August 2019,1,True,NaN,NaN,Otra vez basuroñifilms ha creado otra cagada v...,392,Action,2019-04-25,2019,2010-2018,Vengadores: Endgame,pelicula,Acción|Aventura|Ciencia ficción,sensacine,https://www.sensacine.com/peliculas/pelicula-2...,2026-10-31T14:03:55
2,15003,sensacine-232669,NaN,27 April 2019,10,True,NaN,NaN,Tenia grandes expectativas sobre la pelicula y...,178,Action,2019-04-25,2019,2010-2018,Vengadores: Endgame,pelicula,Acción|Aventura|Ciencia ficción,sensacine,https://www.sensacine.com/peliculas/pelicula-2...,2026-10-31T14:03:55
3,15004,sensacine-232669,NaN,27 April 2019,7,False,NaN,NaN,Excelente película y puedo decir que fue la me...,53,Action,2019-04-25,2019,2010-2018,Vengadores: Endgame,pelicula,Acción|Aventura|Ciencia ficción,sensacine,https://www.sensacine.com/peliculas/pelicula-2...,2026-10-31T14:03:55
4,15005,sensacine-232669,NaN,28 April 2019,9,False,NaN,NaN,Estas en tensión toda la película y no sabes p...,38,Action,2019-04-25,2019,2010-2018,Vengadores: Endgame,pelicula,Acción|Aventura|Ciencia ficción,sensacine,https://www.sensacine.com/peliculas/pelicula-2...,2026-10-31T14:03:55


## 5. Resumen del corpus scrapeado

In [39]:
df = pd.read_csv(sc.RUTA_SALIDA)

print(f"Reseñas: {len(df)} · películas: {df['movie_id'].nunique()}")
print(f"Palabras por reseña: mín {df['n_palabras'].min()}, media {df['n_palabras'].mean():.0f}, máx {df['n_palabras'].max()}")
print(f"Con spoiler: {df['spoiler'].sum()}")
print(f"Fechas de las reseñas: {df['anio'].min()} - {df['anio'].max()}\n")

pd.crosstab(df["genero"], df["periodo"], margins=True, margins_name="Total")

Reseñas: 2000 · películas: 145
Palabras por reseña: mín 30, media 94, máx 400
Con spoiler: 65
Fechas de las reseñas: 2010 - 2026



periodo,2010-2018,Total
genero,,
Action,400,400
Animation,400,400
Comedy,400,400
Drama,400,400
Horror,400,400
Total,2000,2000


In [40]:
# Polaridad con los mismos cortes del Proyecto 1
polaridad = pd.cut(df["calificacion"], bins=[0, 4, 6, 10],
                   labels=["negativa", "intermedia", "positiva"])
pd.crosstab(df["genero"], polaridad, margins=True, margins_name="Total")

calificacion,negativa,intermedia,positiva,Total
genero,,,,
Action,77,91,232,400
Animation,57,76,267,400
Comedy,73,103,224,400
Drama,82,52,266,400
Horror,103,121,176,400
Total,392,443,1165,2000


In [41]:
# Películas que más reseñas aportan
df.groupby(["titulo_obra", "genero"]).size().sort_values(ascending=False).head(10)

titulo_obra                                 genero
Hereditary                                  Drama     25
Los juegos del hambre: En llamas            Action    24
Whiplash                                    Drama     24
Joker                                       Drama     23
Piratas del Caribe: La venganza de Salazar  Action    23
El lobo de Wall Street                      Drama     23
Her                                         Drama     23
Ted                                         Comedy    23
Spider-Man: Homecoming                      Action    23
Midsommar                                   Horror    22
dtype: int64

## 6. Verificación de los requisitos del enunciado (sección 2.3)

In [42]:
por_genero = df["genero"].value_counts()
print("Reseñas por género:", por_genero.to_dict())

assert list(df.columns[:14]) == list(pd.read_csv(sc.RUTA_RESENAS_ES, nrows=0).columns), \
    "Las primeras columnas deben coincidir con resenas_es.csv"
assert df["id"].is_unique and df["id"].min() > pd.read_csv(sc.RUTA_RESENAS_ES, usecols=["id"])["id"].max(), \
    "Los id deben ser únicos y continuar después de resenas_es.csv"
assert df["calificacion"].between(1, 10).all(), "Toda reseña debe tener calificación 1-10"
assert not df["texto_es"].duplicated().any(), "Hay reseñas repetidas"
assert df["user_id"].isna().all(), "No se deben guardar datos de los autores"
assert len(df) <= 5 * METAS_POR_GENERO, "Se pasó de la meta de reseñas"

print(f"\nTotal: {len(df)} de {5 * METAS_POR_GENERO}")
print(f"Géneros con al menos 100 reseñas: {(por_genero >= 100).sum()} de 5")

Reseñas por género: {'Action': 400, 'Animation': 400, 'Comedy': 400, 'Drama': 400, 'Horror': 400}

Total: 2000 de 2000
Géneros con al menos 100 reseñas: 5 de 5
